In [6]:
!git clone https://github.com/venkatsaikondra/CNN_ViT_Hybrid_Vit_Research_Pneumonia_4_Classification.git

fatal: destination path 'CNN_ViT_Hybrid_Vit_Research_Pneumonia_4_Classification' already exists and is not an empty directory.


In [7]:
!pip install transformers[tf]

In [8]:
import os
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from PIL import Image
from transformers import DeiTImageProcessor, DeiTForImageClassification
from sklearn.metrics import classification_report, confusion_matrix
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np

# --- 1. Configuration ---
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
BASE_DIR = '/content/CNN_ViT_Hybrid_Vit_Research_Pneumonia_4_Classification/Train_Test_Split_Vit_Data'
train_data_dir = os.path.join(BASE_DIR, 'train')
val_data_dir = os.path.join(BASE_DIR, 'val')

# --- 2. Initialize DeiT ---
# Using the base-distilled model which includes the distillation token
model_name = "facebook/deit-base-distilled-patch16-224"
feature_extractor_deit = DeiTImageProcessor.from_pretrained(model_name)
model_deit = DeiTForImageClassification.from_pretrained(
    model_name,
    num_labels=4,
    ignore_mismatched_sizes=True
)
model_deit.to(device)

# --- 3. Dataset & Dataloaders ---
class PneumoniaDataset(Dataset):
    def __init__(self, data_dir, feature_extractor):
        self.data_dir = data_dir
        self.feature_extractor = feature_extractor
        self.image_paths = []
        self.labels = []
        self.classes = sorted([d for d in os.listdir(data_dir) if os.path.isdir(os.path.join(data_dir, d))])
        self.class_to_idx = {name: i for i, name in enumerate(self.classes)}
        for class_name in self.classes:
            path = os.path.join(data_dir, class_name)
            for img in os.listdir(path):
                if img.lower().endswith(('.png', '.jpg', '.jpeg')):
                    self.image_paths.append(os.path.join(path, img))
                    self.labels.append(self.class_to_idx[class_name])

    def __len__(self): return len(self.image_paths)
    def __getitem__(self, idx):
        image = Image.open(self.image_paths[idx]).convert('RGB')
        inputs = self.feature_extractor(images=image, return_tensors='pt')
        return inputs['pixel_values'].squeeze(0), torch.tensor(self.labels[idx])

train_dataloader = DataLoader(PneumoniaDataset(train_data_dir, feature_extractor_deit), batch_size=32, shuffle=True)
val_dataloader = DataLoader(PneumoniaDataset(val_data_dir, feature_extractor_deit), batch_size=32, shuffle=False)

# --- 4. Optimizer ---
optimizer = AdamW(model_deit.parameters(), lr=2e-5)
criterion = nn.CrossEntropyLoss()

preprocessor_config.json:   0%|          | 0.00/287 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

pytorch_model.bin:   0%|          | 0.00/349M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

model.safetensors:   0%|          | 0.00/349M [00:00<?, ?B/s]

DeiTForImageClassification LOAD REPORT from: facebook/deit-base-distilled-patch16-224
Key                            | Status     | 
-------------------------------+------------+-
distillation_classifier.weight | UNEXPECTED | 
cls_classifier.bias            | UNEXPECTED | 
cls_classifier.weight          | UNEXPECTED | 
distillation_classifier.bias   | UNEXPECTED | 
classifier.weight              | MISSING    | 
classifier.bias                | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [ ]:
epochs = 25
print(f"Starting DeiT Training on {device}...")

for epoch in range(epochs):
    model_deit.train()
    train_loss, correct, total = 0, 0, 0
    for batch in train_dataloader:
        inputs, labels = batch
        inputs, labels = inputs.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model_deit(inputs).logits
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        train_loss += loss.item()
        _, predicted = outputs.max(1)
        total += labels.size(0)
        correct += predicted.eq(labels).sum().item()

    # Quick Validation
    model_deit.eval()
    val_correct, val_total = 0, 0
    with torch.no_grad():
        for v_batch in val_dataloader:
            v_inputs, v_labels = v_batch
            v_inputs, v_labels = v_inputs.to(device), v_labels.to(device)
            val_outputs = model_deit(v_inputs).logits
            _, v_pred = val_outputs.max(1)
            val_total += v_labels.size(0)
            val_correct += v_pred.eq(v_labels).sum().item()

    print(f"DeiT Epoch {epoch+1}/{epochs} | Loss: {train_loss/len(train_dataloader):.4f} | "
          f"Train Acc: {100.*correct/total:.2f}% | Val Acc: {100.*val_correct/val_total:.2f}%")

Starting DeiT Training on cuda...
DeiT Epoch 1/25 | Loss: 0.4217 | Train Acc: 82.12% | Val Acc: 86.13%
DeiT Epoch 2/25 | Loss: 0.2395 | Train Acc: 90.48% | Val Acc: 90.40%
DeiT Epoch 3/25 | Loss: 0.1135 | Train Acc: 95.73% | Val Acc: 90.49%
DeiT Epoch 4/25 | Loss: 0.0450 | Train Acc: 98.60% | Val Acc: 90.21%
DeiT Epoch 5/25 | Loss: 0.0322 | Train Acc: 98.88% | Val Acc: 91.05%
DeiT Epoch 6/25 | Loss: 0.0231 | Train Acc: 99.30% | Val Acc: 92.67%
DeiT Epoch 7/25 | Loss: 0.0156 | Train Acc: 99.57% | Val Acc: 92.21%
DeiT Epoch 8/25 | Loss: 0.0192 | Train Acc: 99.41% | Val Acc: 92.44%
DeiT Epoch 9/25 | Loss: 0.0170 | Train Acc: 99.48% | Val Acc: 91.88%
DeiT Epoch 10/25 | Loss: 0.0245 | Train Acc: 99.19% | Val Acc: 91.93%
DeiT Epoch 11/25 | Loss: 0.0098 | Train Acc: 99.68% | Val Acc: 92.30%
DeiT Epoch 12/25 | Loss: 0.0123 | Train Acc: 99.59% | Val Acc: 91.05%
DeiT Epoch 13/25 | Loss: 0.0212 | Train Acc: 99.30% | Val Acc: 91.28%
DeiT Epoch 14/25 | Loss: 0.0083 | Train Acc: 99.73% | Val Acc: 91

In [ ]:
model_deit.eval()
all_preds, all_labels = [], []
class_names = ['Covid-19', 'Normal', 'Pneumonia-Bacterial', 'Pneumonia-Viral']

with torch.no_grad():
    for batch in val_dataloader:
        inputs, labels = batch
        inputs, labels = inputs.to(device), labels.to(device)
        outputs = model_deit(inputs).logits
        _, predicted = outputs.max(1)
        all_preds.extend(predicted.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

# Confusion Matrix
plt.figure(figsize=(10, 8))
cm = confusion_matrix(all_labels, all_preds)
sns.heatmap(cm, annot=True, fmt='d', cmap='Greens', xticklabels=class_names, yticklabels=class_names)
plt.title('DeiT Confusion Matrix')
plt.show()

# Classification Report
print("\nDeiT Classification Report:\n")
print(classification_report(all_labels, all_preds, target_names=class_names))

In [ ]:
import numpy as np
import torch
import torch.nn as nn
from PIL import Image
import matplotlib.pyplot as plt

class DeiT_RISE_Explainer(nn.Module):
    def __init__(self, model, input_size=(224, 224), gpu_batch=16):
        super().__init__()
        self.model = model
        self.input_size = input_size
        self.gpu_batch = gpu_batch

    def generate_masks(self, N, s, p1):
        cell_size = np.ceil(np.array(self.input_size) / s)
        up_size = (s + 1) * cell_size
        grid = np.random.rand(N, s, s) < p1
        grid = grid.astype('float32')
        masks = np.empty((N, *self.input_size))
        for i in range(N):
            x = np.random.randint(0, cell_size[0])
            y = np.random.randint(0, cell_size[1])
            # Bilinear upsampling to smooth the masks
            mask_img = Image.fromarray(grid[i]).resize((int(up_size[1]), int(up_size[0])), resample=Image.BILINEAR)
            masks[i] = np.array(mask_img)[x:x+self.input_size[0], y:y+self.input_size[1]]
        return torch.from_numpy(masks).float().to(device)

    def explain(self, x, N=1000, s=8, p1=0.5):
        self.model.eval()
        masks = self.generate_masks(N, s, p1)
        stack = []
        for i in range(0, N, self.gpu_batch):
            m = masks[i:i+self.gpu_batch].unsqueeze(1)
            masked_x = x * m
            with torch.no_grad():
                logits = self.model(masked_x).logits
                stack.append(torch.softmax(logits, dim=1))
        predictions = torch.cat(stack)
        sal = torch.matmul(predictions.data.transpose(0, 1), masks.view(N, -1))
        sal = sal.view(-1, *self.input_size)
        return sal / N / p1

# --- 2. Multi-Class Granularity Visualization for DeiT ---
def visualize_deit_granularity(model, image_path, class_names):
    orig_img = Image.open(image_path).convert('RGB').resize((224, 224))
    # Using the specific DeiT feature extractor
    x = feature_extractor_deit(images=orig_img, return_tensors='pt')['pixel_values'].to(device)

    explainer = DeiT_RISE_Explainer(model)
    saliency_maps = explainer.explain(x)

    plt.figure(figsize=(20, 5))
    plt.subplot(1, 5, 1)
    plt.imshow(orig_img)
    plt.title("Original X-ray")
    plt.axis('off')

    for i in range(4):
        plt.subplot(1, 5, i+2)
        plt.imshow(orig_img)
        # Use a different color map like 'viridis' or 'plasma' to distinguish from Hybrid maps
        plt.imshow(saliency_maps[i].cpu(), cmap='plasma', alpha=0.5)
        plt.title(f"DeiT Focus: {class_names[i]}")
        plt.axis('off')

    plt.show()

# Run the visualization
visualize_deit_granularity(model_deit, some_test_image_path, class_names)